# 📝 파이썬 기초 과제 LV2(응용) — 파일·예외·모듈·클래스

> LV1 개념들을 **조합**하는 실전 문제입니다. 함수·예외·파일·모듈을 함께 씁니다.

## 풀이 방법
1. **답안 셀**에 코드를 채우고, 아래 **자가채점 셀**을 실행해 `✅ 통과!` 를 확인하세요.
2. 막히면 `힌트` 를 펼쳐 보세요.
3. 읽기는 `data/`, 쓰기는 `output/` (없으면 `os.makedirs` 로 생성).

한 단계 올라갑니다!

## 1. 재시도 안전 입력 함수
**배경**: 회원가입 폼에서 잘못 입력하면 몇 번 다시 물어보고, 그래도 안 되면 기본값을 씁니다.

**요구사항**:
- `input_in_range(low, high, default, tries=3)` 함수를 만드세요.
- 숫자를 입력받아 `low ~ high` 범위면 그 값을 돌려줍니다.
- 숫자가 아니거나 범위를 벗어나면 다시 물어봅니다. **최대 `tries` 번** 시도합니다.
- `tries` 번 모두 실패하면 `default` 를 돌려줍니다.

<details><summary>힌트</summary>

```text
접근방법:
- 정해진 횟수만큼만 입력을 다시 받고, 모두 실패하면 기본값을 쓴다.

세부구현:
1. 최대 tries 번 반복한다
   1-1. 입력을 정수로 변환 시도 (숫자가 아니면 ValueError 를 잡아 다시 물어봄)
   1-2. 변환한 값이 low~high 범위 안이면 그 값을 반환
   1-3. 범위를 벗어나면 안내하고 다음 시도로 넘어감
2. 반복이 모두 끝나면(전부 실패) 기본값 default 를 반환
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  (입력을 미리 정해 두고 재시도·범위 검사를 자동 검증합니다)
import builtins
_original_input = builtins.input

_seq = iter(["abc", "999", "7"])          # 숫자아님 -> 범위밖 -> 정상값
builtins.input = lambda *a: next(_seq)
assert input_in_range(1, 10, default=0) == 7, "잘못된 입력은 넘기고, 올바른 값이 나오면 그 값을 돌려줘야 해요"

_seq = iter(["999", "888", "777"])        # 3번 모두 범위 밖
builtins.input = lambda *a: next(_seq)
assert input_in_range(1, 10, default=5, tries=3) == 5, "tries 번 모두 실패하면 기본값을 돌려줘야 해요"

builtins.input = _original_input
print("✅ 문제1 통과!")

In [ ]:
# 직접 입력하며 확인해 보려면 아래 두 줄의 주석을 풀고 실행하세요.
# 범위(1~10)를 벗어난 숫자를 넣으면 다시 물어보고, 3번 실패하면 기본값을 씁니다.
# value = input_in_range(1, 10, default=5)
# print("결정된 값:", value)

## 2. 우수 학생 필터 → 새 CSV 저장
**배경**: 전체 성적표에서 조건에 맞는 사람만 뽑아 별도 파일로 만드는 건 흔한 작업이에요.

**요구사항**:
- `filter_top(in_path, out_path)` 함수를 만드세요.
- `in_path`(students.csv)를 읽어 **평균 80점 이상**인 학생만 골라
  `out_path` 에 `이름,평균` 헤더의 CSV 로 저장하세요.
- 저장한 학생 수를 돌려주세요.

<details><summary>힌트</summary>

```text
접근방법:
- 원본 CSV를 읽어 평균 80 이상만 골라 새 CSV로 저장한다.

세부구현:
1. in_path 를 DictReader 로 읽어 행 목록을 만든다
2. 각 행의 세 과목 평균을 계산해, 80 이상이면 [이름, 반올림한 평균] 을 새 목록에 추가
3. out_path 의 폴더를 준비하고, 헤더(이름,평균)를 먼저 쓴 뒤 모은 행들을 기록
4. 저장한 학생 수를 반환
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import csv
count = filter_top("data/students.csv", "output/우수학생.csv")
with open("output/우수학생.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
assert len(rows) == count, "저장한 행 수와 반환값이 같아야 해요"
assert all(float(r["평균"]) >= 80 for r in rows), "평균 80 이상만 있어야 해요"
assert count >= 1
print("✅ 문제2 통과! (", count, "명)")

## 3. JSON → CSV 변환기
**배경**: JSON 으로 받은 데이터를 엑셀에서 열 수 있게 CSV 로 바꿔 달라는 요청은 아주 흔해요.

**요구사항**:
- `json_to_csv(json_path, csv_path)` 함수를 만드세요.
- `menu.json` 을 읽어 각 메뉴를 CSV 한 줄로 저장합니다.
- CSV 헤더는 `이름,가격,종류,품절` 로 하세요.
- 변환한 메뉴 개수를 돌려주세요.

<details><summary>힌트</summary>

```text
접근방법:
- JSON 메뉴 목록을 읽어 각 메뉴를 CSV 한 줄로 옮긴다.

세부구현:
1. json_path 를 읽어 메뉴 목록(딕셔너리들의 리스트)을 얻는다
2. csv_path 의 폴더를 준비하고, 헤더(이름,가격,종류,품절)를 먼저 쓴다
3. 메뉴 하나마다 네 값을 꺼내 한 줄씩 기록
4. 변환한 메뉴 개수를 반환
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import csv
json_to_csv("data/menu.json", "output/menu.csv")
with open("output/menu.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
assert len(rows) == 9, "메뉴 9개가 저장돼야 해요"
assert rows[0]["이름"] == "아메리카노"
print("✅ 문제3 통과!")

## 4. 나만의 유틸 모듈 만들기
**배경**: 자주 쓰는 함수는 모듈(`.py`)로 빼 두면 여러 프로그램에서 재사용할 수 있어요.

**요구사항**:
- 아래 `[제공 코드]` 셀이 `mytools.py` 에 `won(n)`(숫자를 `"1,000원"` 형태로) 함수를 만들어 둡니다.
- 여기에 **`is_even(n)`**(짝수면 True) 함수를 **추가**해 모듈을 완성하고, `import` 해서 두 함수를 사용하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 모듈 파일에 함수를 마저 채워 넣고 import 해서 쓴다.

세부구현:
1. mytools.py 파일에 won 과 is_even 두 함수를 모두 담아 다시 저장 (is_even 은 n 을 2로 나눈 나머지가 0인지로 짝수를 판단)
2. 현재 폴더를 모듈 검색 경로에 추가한 뒤 mytools 를 import
3. won 과 is_even 을 호출해 결과를 출력

캐싱 주의: 한 번 import 한 모듈은 파일을 고쳐도 자동 반영되지 않아요. 수정했다면 커널 재시작(Restart) 후 다시 import 하세요.
```

</details>

In [ ]:
# [제공 코드] 먼저 기본 모듈 파일을 하나 만들어 둡니다. (won 함수 하나)
base_module = '''# mytools.py — 나만의 유틸 모듈
def won(n):
    return format(n, ",") + "원"
'''
with open("mytools.py", "w", encoding="utf-8") as f:
    f.write(base_module)
print("mytools.py 를 만들었어요! 여기에 is_even 함수를 추가해 완성해 보세요.")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import mytools
assert mytools.won(1000) == "1,000원"
assert mytools.is_even(4) is True
assert mytools.is_even(7) is False
print("✅ 문제4 통과!")

## 5. 가계부 월별·분류별 집계
**배경**: "이번 달 교통비 얼마 썼지?" 같은 질문에 답하려면 날짜·분류별로 합계를 내야 해요.

**요구사항**:
- `summarize(path)` 함수를 만드세요.
- `가계부_샘플.csv` 를 읽어 **월별 → 분류별 합계**를 담은 딕셔너리를 돌려줍니다.
- 결과 형태: `{ "2026-05": {"식비": 80000, "교통": 12700, ...}, "2026-06": {...} }`

<details><summary>힌트</summary>

```text
접근방법:
- 가계부 CSV를 읽어 '월 → 분류 → 합계' 중첩 딕셔너리를 만든다.

세부구현:
1. csv 를 DictReader 로 읽어 행 목록을 만든다
2. 각 행에서 날짜의 앞 7글자로 월을 구하고, 분류와 금액(정수)을 꺼낸다
3. 그 월이 처음 등장하면 빈 딕셔너리로 초기화한 뒤, 월별-분류별 합계에 금액을 누적
4. 완성한 중첩 딕셔너리를 반환
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
table = summarize("data/가계부_샘플.csv")
assert "2026-05" in table and "2026-06" in table and "2026-07" in table
assert table["2026-05"]["교통"] == 1400 + 1500 + 9800   # 5월 교통비 합계
print("✅ 문제5 통과!")

## 6. raise 로 값 검증하기
**배경**: 잘못된 값(음수 나이 등)은 조용히 넘기기보다 **분명하게 에러로 알려야** 버그를 빨리 잡습니다.

**요구사항**:
- `check_age(age)` 함수를 만드세요.
- `age` 가 0 미만이거나 150 초과이면 `ValueError` 를 **raise** 하세요.
- 정상 범위면 `age` 를 그대로 돌려줍니다.
- 그리고 여러 나이 값을 `try/except` 로 안전하게 호출하는 예시를 보여 주세요.

<details><summary>힌트</summary>

```text
접근방법:
- 규칙에 어긋난 값이면 함수가 직접 예외를 일으키고, 호출하는 쪽이 그것을 처리한다.

세부구현:
1. age 가 0 미만이거나 150 초과이면 ValueError 를 raise
2. 정상 범위면 age 를 그대로 반환
3. 호출하는 쪽에서는 여러 나이 값을 for 로 돌며 try/except 로 감싸, 예외가 나면 안내를 출력
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert check_age(30) == 30

neg_caught = False
try:
    check_age(-1)
except ValueError:
    neg_caught = True
assert neg_caught, "음수 나이는 ValueError 를 내야 해요"

big_caught = False
try:
    check_age(999)
except ValueError:
    big_caught = True
assert big_caught, "너무 큰 나이도 ValueError 를 내야 해요"
print("✅ 문제6 통과!")

## 7. 도형 유틸 패키지 만들기
**배경**: 자주 쓰는 함수는 폴더(패키지)로 묶어 두면 여러 프로그램에서 재사용할 수 있어요. 교안에서 만든 `shop/` 처럼 나만의 패키지를 만들어 봅시다.

**요구사항**:
- `geometry/` 폴더에 모듈 두 개와 `__init__.py` 를 만들어 패키지로 구성하세요.
  - `geometry/area.py` : `circle_area(r)`(원 넓이 `math.pi * r * r`), `rect_area(w, h)`(직사각형 넓이 `w * h`)
  - `geometry/convert.py` : `cm_to_m(cm)`(센티미터를 미터로, `cm / 100`)
  - `geometry/__init__.py` : **이 파일이 있어야 폴더가 패키지가 돼요.** 내용은 **비워 둬도 됩니다.**
- 그 뒤 교안의 `shop.menu.show_menu()` 처럼 **모듈 경로로** 불러와 쓸 수 있어야 합니다.
  - `geometry.area.circle_area(...)`, `geometry.area.rect_area(...)`, `geometry.convert.cm_to_m(...)`

<details><summary>힌트</summary>

```text
접근방법:
- 폴더를 만들고 기능별로 모듈 파일(.py)을 나눠 쓴 뒤, __init__.py 로 패키지가 되게 한다.
  교안의 shop 패키지(menu.py / pay.py / __init__.py)와 똑같은 구조예요.

세부구현:
1. geometry 폴더를 만든다 (os.makedirs, 이미 있어도 에러 안 나게)
2. area.py 에 원 넓이·직사각형 넓이 함수를, convert.py 에 센티미터->미터 함수를 파일로 써 넣는다
3. __init__.py 파일도 만든다 (비어 있어도 됨 — 이 파일이 있으면 폴더가 패키지가 됨)
4. sys.path.insert(0, os.getcwd()) 로 현재 폴더를 알려 준 뒤
   import geometry.area / import geometry.convert 로 불러와
   geometry.area.circle_area(...) 처럼 모듈 경로로 호출한다

캐싱 주의: 한 번 import 한 패키지는 파일을 고쳐도 자동 반영되지 않아요. 수정했다면 커널 재시작(Restart) 후 다시 import 하세요.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import sys, os
sys.path.insert(0, os.getcwd())
import geometry.area
import geometry.convert
assert geometry.area.rect_area(3, 4) == 12, "직사각형 넓이는 가로×세로"
assert round(geometry.area.circle_area(2), 2) == 12.57, "원 넓이는 math.pi * r * r"
assert geometry.convert.cm_to_m(250) == 2.5, "cm_to_m 은 cm / 100"
print("✅ 문제7 통과!")

## 8. 로그인 기록하고 횟수 세기 (이어 쓰기 + 다시 읽기 + 집계)
**배경**: 접속 로그를 파일에 쌓아 두고, 그 파일을 **다시 읽어** "이 사람이 몇 번 로그인했는지" 세어 봐요. 이어 쓰기·읽기·집계를 한 번에 엮는 문제예요.

**요구사항**:
- `login_and_count(path, name)` 함수를 만드세요.
- 먼저 `path` 파일에 `name` 을 **한 줄 이어 씁니다**(추가 모드 `"a"`, 폴더 없으면 `os.makedirs`).
- 그런 다음 **파일 전체를 다시 읽어**, `name` 과 같은 줄이 **몇 개인지**(그 사람의 누적 로그인 횟수)를 세어 반환합니다.

**예시**
```
login_and_count("output/logins.log", "민준")  ->  1
login_and_count("output/logins.log", "서연")  ->  1
login_and_count("output/logins.log", "민준")  ->  2   (민준 두 번째 로그인)
```

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 이름을 파일에 이어 쓰고(a 모드), 그 파일을 통째로 다시 읽어 같은 이름의 줄 수를 센다

세부구현:
1. os.makedirs 로 상위 폴더를 준비하고, "a" 모드로 열어 name + 줄바꿈("\n")을 write
2. 파일을 다시 열어 모든 줄을 읽는다 (read().splitlines(), 빈 줄 제외)
3. 그 줄들 중 name 과 같은 것의 개수를 세어 반환 (리스트의 count 활용)
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import os
log_path = "output/logins_test.log"
if os.path.exists(log_path):
    os.remove(log_path)
assert login_and_count(log_path, "민준") == 1
assert login_and_count(log_path, "서연") == 1
assert login_and_count(log_path, "민준") == 2, "민준의 두 번째 로그인은 2 여야 해요"
assert login_and_count(log_path, "민준") == 3
assert login_and_count(log_path, "서연") == 2
print("✅ 문제8 통과!")

## 9. 장바구니 클래스 (상태를 가진 클래스)
**배경**: 클래스의 진짜 힘은 **객체가 자기 상태(속성)를 기억하고 메서드로 바꾸는** 데 있어요. 상품을 담을수록 목록과 합계가 쌓이는 장바구니를 만들어 봅시다.

**요구사항**:
- `Cart` 클래스를 만드세요.
  - `__init__(self)` : `self.items` 를 빈 리스트 `[]`, `self.total` 을 `0` 으로 시작
  - `add(self, name, price)` : `items` 에 상품 이름을 추가하고, `total` 에 가격을 더함
  - `count(self)` : 담긴 상품 **개수**를 반환

**예시**
```
c = Cart()
c.add("사과", 1000)
c.add("바나나", 1500)
c.count()   ->  2
c.total     ->  2500
c.items     ->  ["사과", "바나나"]
```

<details><summary>힌트</summary>

```text
접근방법:
- __init__ 에서 빈 리스트와 합계 0 을 준비하고, add 가 호출될 때마다 목록·합계를 갱신한다

세부구현:
1. class Cart: 를 정의
2. def __init__(self): 안에서 self.items = [], self.total = 0
3. def add(self, name, price): 는 self.items 에 name 을 append 하고 self.total += price
4. def count(self): 는 len(self.items) 를 return
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
c = Cart()
c.add("사과", 1000)
c.add("바나나", 1500)
assert c.count() == 2, "담은 상품 개수"
assert c.total == 2500, "가격 합계가 쌓여야 해요"
assert c.items == ["사과", "바나나"], "담은 순서대로 이름이 들어가야 해요"
print("✅ 문제9 통과!")